# Day 1 · Module 4: Worktrees (Two Caching Strategies, Non-Test Decision)

**Objective:** run two legitimate implementations of the same plan in parallel git worktrees off one known commit, then decide which one ships using a criterion that "both pass tests" cannot answer.


## Relevant Claude Code Docs
Review these before starting the module:
- [Run parallel sessions with worktrees](https://code.claude.com/docs/en/worktrees)
- [Run agents in parallel](https://code.claude.com/docs/en/agents)
- [Set up Claude Code in a monorepo or large codebase](https://code.claude.com/docs/en/large-codebases)

## 1 · The idea

Git worktrees let you check out the same commit into multiple directories at once, each with its own working tree but sharing one `.git`. That makes it cheap to run two different implementations of the same plan side by side, from the identical starting point, instead of stashing and switching branches in one directory and hoping you compared like with like.

The harder part isn't the mechanics of worktrees — it's what you do once both implementations exist and both pass their tests. Tests check that an implementation satisfies stated acceptance criteria; they rarely tell you which implementation is *better* to operate. That decision usually turns on something tests don't measure at all: how much blast radius a change has, how it fails, or what it silently trades away — and at a bank, what a stale number can cause a customer or a regulator to do.


### Grounding

`analytics.summary()` recomputes from every row in `ledger` on every call, which is slow for active accounts. `reference/m4/plan.md` deliberately does not choose a caching strategy — it evaluates two, separately, against the same acceptance criteria:

- **Write-through:** update a cached summary on every posting. The write hook lands in the hot posting path — `posting.post()` calling `ledger.record_entry()` — so it inherits that path's existing non-atomic commit issue (authorize and record are already separate commits) and adds new blast radius: anything slow, blocking, or that can raise in the cache-update hook now risks every transfer posting, not just analytics reads.
- **Cache-aside + TTL:** compute lazily, cache the result, expire or invalidate after a bound. This leaves the hot posting path alone entirely, but trades that safety for staleness — `summary()` can return an outdated entry count or region breakdown for up to the TTL window.

Both can satisfy the plan's acceptance criteria and both can pass their own tests. The plan's own "Risks" section names the non-test criteria that should decide between them: hot-path blast radius and staleness. A third and fourth belong in the decision too — invalidation complexity (how easy is it to get wrong, silently, per account), and the banking-specific one: regulatory/operational tolerance for a stale balance. A stale account summary isn't just a slow dashboard — it can drive an overdraft decision or let a duplicate spend through, which makes this a compliance question, not only a latency one.


### Setup check

Run the cell below once per session. It makes sure `contoso` is importable and prints the resolved project root — you'll need `proj` for the grounding check, the exercise, and the self-check.


In [ ]:
import subprocess, sys, os
# Ensure src is importable and the sandbox is healthy before you start.
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) in ("day1","day2") else None
root = subprocess.run(["git","rev-parse","--show-toplevel"], capture_output=True, text=True)
proj = root.stdout.strip() or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(proj, "src"))
print("project root:", proj)
try:
    import contoso
    print("contoso imported OK")
except Exception as e:
    print("Could not import contoso:", e)
    print("Run `uv sync` in the repo root, then restart the kernel.")


With `proj` resolved, confirm the shape of the hot path and the recompute directly against the real source:


In [ ]:
import pathlib
src = pathlib.Path(proj) / "src" / "contoso"
an = (src / "analytics.py").read_text()
post = (src / "posting.py").read_text()
print("analytics.summary() recomputes from every ledger row:", "def summary" in an and "ledger" in an)
print("posting.post() is the hot path and calls ledger.record_entry:", "def post" in post and "ledger.record_entry" in post)


## 2 · Your exercise

> ⏱ **Time-box: ~15 min per session.** Stop when the timer fires, even mid-implementation — the exercise is about comparing two partial, honest attempts, not about finishing either one.

Work through these steps in order:

1. Copy `reference/m4/plan.md` to `artifacts/m4/plan.md` — this is the canned plan that intentionally leaves the caching strategy open; you feed the same plan to both approaches below.
2. Create two worktrees off the current commit:
   ```
   git worktree add ../contoso-cache-writethrough HEAD
   git worktree add ../contoso-cache-aside HEAD
   ```
   (Why not just stash and switch branches in one directory? You'd be comparing two implementations that never coexist — one is always stashed away while the other runs, so "same starting point" is an assertion, not a fact you can check. Two worktrees off the same `HEAD` make "identical start" verifiable with `git worktree list`, and let both sessions' uncommitted mess exist at the same time without touching each other.)
3. In `../contoso-cache-writethrough`, start a Claude Code session constrained to the write-through approach only. In `../contoso-cache-aside`, start a separate Claude Code session constrained to the cache-aside + TTL approach only. Time-box each session to the ~15 minutes above and have each implement the plan and run its own tests.
4. Back in this main worktree, write `artifacts/m4/comparison.md` comparing the two: lines changed, files touched, test results, how each handles the edge cases in the plan's acceptance criteria (an account with zero ledger entries, an entry posted for a different `account_id`), and a ship recommendation that names which non-test criterion decided it.
5. Clean up both worktrees so they don't strand and pollute later modules:
   ```
   git worktree remove ../contoso-cache-writethrough
   git worktree remove ../contoso-cache-aside
   git worktree prune
   ```
   (`prune` covers the case where a worktree directory was deleted by hand instead of via `git worktree remove` — it clears the now-dangling metadata `git` would otherwise keep pointing at it.)


### What good looks like

A good `comparison.md` reports both approaches' lines changed, files touched, and test results — and then decides using something tests did not check. The candidate non-test criteria are: the write-through approach's blast radius on the hot posting path (`posting.post()`), the cache-aside approach's staleness window and invalidation complexity, and the banking-specific one — how much regulatory/operational tolerance the bank has for a stale balance driving an overdraft or duplicate-spend decision. If "both pass tests" is the only reason given for the ship recommendation, the module was missed — that sentence is where the real work should start, not end.

Common failure modes:
- Comparing diff size or test pass/fail alone and stopping there.
- Recommending write-through without measuring or at least naming what happens if the cache-update hook is slow or raises inside `posting.post()`.
- Recommending cache-aside without stating the TTL's staleness bound explicitly, so "eventually consistent" never gets pinned down to a number a regulator or an on-call operator could act on.


### Fast-finisher

Ask each Claude Code session, from inside its own worktree, to name the one thing about the *other* strategy that would concern it most if it were the on-call operator for ContosoBank tonight. Compare those two answers to what you wrote in `comparison.md` — did either session name a risk you didn't?


### Verify your work

This checklist is advisory, not a gate — it lists the current worktrees and reflects back what it finds in `artifacts/m4/comparison.md`. It's safe to run at any point, including before you've created any worktrees, and again after cleanup to confirm both were removed.


In [ ]:
import pathlib, subprocess
wl = subprocess.run(["git","worktree","list"], cwd=proj, capture_output=True, text=True).stdout
print("worktrees:\n" + wl)
comp = pathlib.Path(proj) / "artifacts" / "m4" / "comparison.md"
if comp.exists():
    t = comp.read_text().lower()
    print(f"[x] comparison.md present ({len(t.split())} words)")
    print(f"  [{'x' if 'write-through' in t and ('cache-aside' in t or 'ttl' in t) else ' '}] compares BOTH approaches?")
    print(f"  [{'x' if 'ship' in t or 'recommend' in t else ' '}] states a ship recommendation?")
    decides = any(w in t for w in ('blast','coupling','complexity','staleness','operational','invalidation','regulatory','overdraft'))
    print(f"  [{'yes' if decides else 'CHECK THIS'}] decides on a NON-TEST criterion (blast radius / staleness / invalidation / regulatory tolerance)?")
else:
    print("[ ] artifacts/m4/comparison.md missing")


## 3 · Debrief

- Why start both implementations from the same known commit via worktrees, rather than two branches switched back and forth in one directory?
- What made "both pass tests" insufficient as a ship decision here — and is that true of every caching change, or specific to a hot path like `posting.post()`?
- If you had to operate whichever approach you didn't recommend, what would you want monitored on day one?
- Which of the four criteria — blast radius, staleness, invalidation complexity, regulatory/operational tolerance — actually decided your recommendation, and would a different bank's risk appetite have flipped it?


---
### Take-home

Worktrees make "same commit, two implementations" cheap enough to actually do instead of just discuss. Once both exist and both pass tests, the decision that matters is usually a non-test one — blast radius, staleness, invalidation complexity, or a bank's tolerance for a stale balance — and naming it explicitly in the comparison is the whole point of the exercise. Cleaning up both worktrees when you're done is part of the exercise too, not an afterthought.

(Related concept: Module 5 orchestrates a whole team of subagents at once, rather than two isolated sessions compared after the fact.)
